# Week 05 실습 · 함수 · 모듈 · 테스트 가능한 코드

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week05/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. 매개변수·기본값·키워드 인자·여러 값 반환·docstring·타입 힌트로 **읽기 쉬운 함수**를 만듭니다.
2. 지역/전역 범위와 **순수 함수**(print 대신 return)의 장점을 이해합니다.
3. `assert`로 테스트 함수를 만들고, 잘못된 입력에는 `raise ValueError`로 알립니다.
4. `%%writefile`로 **모듈 파일**을 만들어 `import`해서 재사용합니다.
5. AI 코드의 **가변 기본값 버그**를 테스트로 찾아냅니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | Level 1 함수의 구성 요소 | 20분 |
| 2 | Level 1 범위와 순수 함수 | 15분 |
| 3 | Level 1 테스트와 잘못된 입력 | 20분 |
| 4 | Level 2 바꾸기 | 10분 |
| 5 | Level 3 성적 계산 모듈 만들기 | 30분 |
| 6 | Level 4 AI 함수 검증 | 15분 |
| 7 | 정리 | 10분 |

---
## 1. Level 1 따라하기: 함수의 구성 요소

```python
def 함수이름(매개변수1, 매개변수2=기본값) -> 반환자료형:
    """한 줄 설명 (docstring)"""
    처리
    return 결과
```

In [ ]:
def weighted_total(midterm: float, final: float, w_mid: float = 0.4) -> float:
    """중간·기말 점수의 가중 합계를 돌려준다. 기말 가중치는 1 - w_mid."""
    w_final = 1 - w_mid
    return midterm * w_mid + final * w_final


print(weighted_total(80, 90))                # 기본값 w_mid=0.4 사용
print(weighted_total(80, 90, 0.5))           # 위치 인자
print(weighted_total(final=90, midterm=80))  # 키워드 인자: 순서를 바꿔도 된다
print(weighted_total(80, 90, w_mid=0.3))

In [ ]:
help(weighted_total)     # docstring과 타입 힌트가 설명서가 된다

### 여러 값 돌려주기

`return a, b`는 튜플 하나를 돌려주는 것이고, 받는 쪽에서 언패킹합니다.

In [ ]:
def describe(scores):
    """점수 리스트의 (평균, 최저, 최고)를 돌려준다."""
    avg = sum(scores) / len(scores)
    return round(avg, 1), min(scores), max(scores)


avg, low, high = describe([72, 95, 88, 59])
print(f"평균 {avg}, 최저 {low}, 최고 {high}")

### ❓ 예상해 보기: `return`이 없는 함수

In [ ]:
def show_total(midterm, final):
    print("합계:", midterm + final)

result = show_total(80, 90)
print("돌려받은 값:", result)

`return`이 없으면 함수는 `None`을 돌려줍니다. `print`는 **화면에 보여 줄 뿐** 값을 넘겨주지 않습니다.
→ 계산 함수는 **`return`으로 값을 돌려주고**, 출력은 함수 밖에서 합니다. 그래야 결과를 다시 쓰고 테스트할 수 있습니다.

---
## 2. 범위(scope)와 순수 함수

- 함수 안에서 만든 변수는 **지역 변수**: 함수가 끝나면 사라집니다.
- 함수 밖의 변수(전역)를 함수 안에서 **바꾸려 하면** 문제가 생깁니다.

In [ ]:
bonus = 5

def add_bonus(score):
    result = score + bonus       # 전역 변수를 읽는 것은 가능
    return result

print(add_bonus(80))
# print(result)  ← NameError: 함수 안의 지역 변수는 밖에서 볼 수 없다

In [ ]:
counter = 0

def increase():
    counter = counter + 1        # 전역을 바꾸려 하면?
    return counter

try:
    increase()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

`global` 키워드로 해결할 수도 있지만, **전역 변수에 기대는 함수는 테스트하기 어렵습니다.** 필요한 값은 매개변수로 받고, 결과는 `return`으로 돌려주는 **순수 함수**를 기본으로 합니다.

| 순수 함수 | 부수효과가 있는 함수 |
|---|---|
| 같은 입력 → 항상 같은 출력 | 전역 변수·파일·화면 등에 따라 달라짐 |
| 입력을 바꾸지 않음 | 입력 리스트를 수정하기도 함 |
| `assert f(2) == 4`로 바로 테스트 | 테스트 전에 상태를 준비해야 함 |

In [ ]:
def increase_pure(counter):
    return counter + 1

c = 0
c = increase_pure(c)
c = increase_pure(c)
print(c)

---
## 3. 테스트와 잘못된 입력

### 잘못된 입력은 `raise`로 알린다

이상한 값을 받았을 때 조용히 엉뚱한 결과를 내는 것보다 **즉시 오류를 내는 것**이 낫습니다. (6주차에 `try/except`로 받아 처리합니다.)

In [ ]:
def letter_grade(total: float) -> str:
    """0~100점 총점을 A~F 등급으로 바꾼다. 범위를 벗어나면 ValueError."""
    if not 0 <= total <= 100:
        raise ValueError(f"총점은 0~100이어야 합니다: {total}")
    if total >= 90:
        return "A"
    if total >= 80:
        return "B"
    if total >= 70:
        return "C"
    if total >= 60:
        return "D"
    return "F"


print(letter_grade(85.5))
try:
    letter_grade(105)
except ValueError as e:
    print("ValueError:", e)

### 테스트 함수 만들기

`test_`로 시작하는 함수 안에 `assert`를 모아 두면, 코드를 고칠 때마다 한 번에 다시 확인할 수 있습니다.
좋은 테스트는 **정상 · 경계 · 잘못된 입력**을 모두 포함합니다.

In [ ]:
def test_letter_grade():
    # 정상
    assert letter_grade(95) == "A"
    assert letter_grade(75) == "C"
    # 경계
    assert letter_grade(90) == "A"
    assert letter_grade(89.99) == "B"
    assert letter_grade(60) == "D"
    assert letter_grade(0) == "F"
    assert letter_grade(100) == "A"
    # 잘못된 입력은 ValueError
    for bad in [-1, 100.1]:
        try:
            letter_grade(bad)
            assert False, f"{bad}에서 ValueError가 나야 함"
        except ValueError:
            pass


def test_weighted_total():
    assert weighted_total(80, 90) == 86.0
    assert weighted_total(100, 100) == 100
    assert weighted_total(0, 0) == 0
    assert abs(weighted_total(70, 85, 0.3) - 80.5) < 1e-9


test_letter_grade()
test_weighted_total()
print("모든 테스트 통과 ✅")

### 작은 테스트 실행기

테스트가 많아지면 **이름이 `test_`로 시작하는 함수를 모두 찾아 실행**하는 도구가 편합니다. (실무에서는 `pytest`가 이 일을 합니다.)

In [ ]:
def run_tests(namespace):
    passed = failed = 0
    for name, obj in list(namespace.items()):
        if name.startswith("test_") and callable(obj):
            try:
                obj()
                print(f"✅ {name}")
                passed += 1
            except AssertionError as e:
                print(f"❌ {name}: {e}")
                failed += 1
    print(f"\n통과 {passed}개, 실패 {failed}개")

run_tests(globals())

---
## 4. Level 2 바꾸기

### ✏️ 직접 해보기 1

1. `weighted_total`에 **과제 점수와 과제 가중치** 매개변수를 추가해 보세요. (중간 0.3, 기말 0.4, 과제 0.3)
2. 기존 호출 `weighted_total(80, 90)`이 **여전히 같은 결과**를 내도록 기본값을 정하세요.
3. 바꾼 뒤 `run_tests(globals())`를 다시 실행해 기존 테스트가 깨지지 않았는지 확인합니다.

In [ ]:
def weighted_total_v2(midterm, final, assignment=0, w_mid=0.4, w_final=0.6, w_assign=0.0):
    """중간·기말·과제의 가중 합계. 기본값으로 호출하면 기존 weighted_total과 같다."""
    return midterm * w_mid + final * w_final + assignment * w_assign


print(weighted_total_v2(80, 90))                                     # 기존과 같은가?
print(weighted_total_v2(80, 90, 70, w_mid=0.3, w_final=0.4, w_assign=0.3))

### ✏️ 직접 해보기 2

`test_letter_grade`에 여러분이 생각한 경계값 테스트를 **두 개 더** 추가하고 실행해 보세요.

---
## 5. Level 3 적용하기: 성적 계산 모듈 만들기

노트북에서 만든 함수를 다른 노트북이나 프로그램에서도 쓰려면 **모듈(.py 파일)** 로 저장합니다.
Colab에서는 `%%writefile 파일이름.py`로 셀 내용을 파일로 저장할 수 있습니다.

**성적 규정 (가상)**

- 총점 = 중간 30% + 기말 40% + 과제 30% (소수 첫째 자리 반올림)
- 출석이 16회 중 12회 미만이면 총점과 관계없이 **F**
- 과제 미제출(`None`)은 0점으로 계산

In [ ]:
%%writefile grade_utils.py
"""성적 계산 함수 모음 (AI활용코딩연습 5주차)."""

WEIGHTS = {"midterm": 0.3, "final": 0.4, "assignment": 0.3}
TOTAL_SESSIONS = 16
MIN_ATTENDANCE = 12


def weighted_total(midterm, final, assignment, weights=None):
    """가중 합계를 소수 첫째 자리까지 돌려준다. assignment가 None이면 0점."""
    w = weights or WEIGHTS
    if abs(sum(w.values()) - 1) > 1e-9:
        raise ValueError("가중치 합은 1이어야 합니다")
    assignment = 0 if assignment is None else assignment
    total = midterm * w["midterm"] + final * w["final"] + assignment * w["assignment"]
    return round(total, 1)


def letter_grade(total):
    """0~100 총점을 A~F로 바꾼다."""
    if not 0 <= total <= 100:
        raise ValueError(f"총점 범위 오류: {total}")
    for cut, letter in [(90, "A"), (80, "B"), (70, "C"), (60, "D")]:
        if total >= cut:
            return letter
    return "F"


def attendance_ok(attended, total_sessions=TOTAL_SESSIONS, minimum=MIN_ATTENDANCE):
    """출석 횟수가 기준 이상인지 돌려준다."""
    if not 0 <= attended <= total_sessions:
        raise ValueError(f"출석 횟수 오류: {attended}")
    return attended >= minimum


def final_grade(record):
    """학생 레코드(dict)로 (총점, 최종 등급)을 돌려준다. 출석 미달이면 F."""
    total = weighted_total(record["midterm"], record["final"], record.get("assignment"))
    if not attendance_ok(record["attendance"]):
        return total, "F"
    return total, letter_grade(total)


if __name__ == "__main__":
    # 파일을 직접 실행할 때만 동작 (import할 때는 실행되지 않음)
    print(final_grade({"midterm": 80, "final": 90, "assignment": 85, "attendance": 15}))

In [ ]:
import importlib
import grade_utils
importlib.reload(grade_utils)    # 파일을 고친 뒤 다시 불러올 때 필요

from grade_utils import final_grade, weighted_total as wt

print(grade_utils.WEIGHTS)
print(wt(80, 90, 85))
print(final_grade({"midterm": 80, "final": 90, "assignment": 85, "attendance": 15}))
print(final_grade({"midterm": 95, "final": 98, "assignment": 100, "attendance": 10}))

In [ ]:
!python grade_utils.py

### 모듈 테스트하기

In [ ]:
def test_module_weighted_total():
    assert grade_utils.weighted_total(80, 90, 85) == 85.5
    assert grade_utils.weighted_total(80, 90, None) == 60.0          # 미제출 = 0점
    assert grade_utils.weighted_total(100, 100, 100) == 100.0


def test_module_final_grade():
    assert final_grade({"midterm": 80, "final": 90, "assignment": 85, "attendance": 15}) == (85.5, "B")
    assert final_grade({"midterm": 95, "final": 98, "assignment": 100, "attendance": 11}) == (97.7, "F")
    assert final_grade({"midterm": 95, "final": 98, "assignment": 100, "attendance": 12})[1] == "A"


def test_module_bad_weights():
    try:
        grade_utils.weighted_total(80, 90, 85, {"midterm": 0.5, "final": 0.5, "assignment": 0.5})
        assert False, "가중치 합이 1이 아니면 ValueError"
    except ValueError:
        pass


run_tests({k: v for k, v in globals().items() if k.startswith("test_module")})

In [ ]:
# 여러 학생에게 적용하기: 함수 덕분에 반복문이 단순해진다
students = [
    {"name": "김민준", "midterm": 78, "final": 85, "assignment": 90, "attendance": 15},
    {"name": "이서연", "midterm": 92, "final": 88, "assignment": None, "attendance": 16},
    {"name": "박지호", "midterm": 95, "final": 97, "assignment": 99, "attendance": 9},
    {"name": "최하은", "midterm": 55, "final": 62, "assignment": 70, "attendance": 14},
]
for s in students:
    total, grade = final_grade(s)
    print(f"{s['name']:<5} 총점 {total:>5}  등급 {grade}")

---
## 6. Level 4 AI와 함께: 가변 기본값의 함정

"학생 레코드를 목록에 추가하는 함수, 목록을 안 주면 새 목록을 만들어서"라고 요청해 받은 코드입니다.

In [ ]:
def add_record_ai(record, records=[]):
    records.append(record)
    return records

class_a = add_record_ai({"name": "김민준"})
class_b = add_record_ai({"name": "이서연"})     # 새 목록을 기대했다
print("class_a:", class_a)
print("class_b:", class_b)
print("같은 리스트인가?", class_a is class_b)

### 🔎 원인

기본값 `[]`는 **함수를 정의할 때 딱 한 번** 만들어지고, 이후 호출에서 **계속 같은 리스트**가 재사용됩니다.
그래서 "새 목록"을 기대한 두 호출이 같은 리스트를 공유합니다.

### 🛠️ 해결: 기본값은 `None`, 함수 안에서 새로 만들기

In [ ]:
def add_record(record, records=None):
    if records is None:
        records = []
    records.append(record)
    return records


def test_add_record_independent():
    a = add_record({"name": "김민준"})
    b = add_record({"name": "이서연"})
    assert a == [{"name": "김민준"}]
    assert b == [{"name": "이서연"}]
    assert a is not b


test_add_record_independent()
print("가변 기본값 버그 수정 확인 ✅")

### 📝 AI에게 함수를 요청할 때

```text
다음 규칙을 지키는 Python 함수를 작성해 주세요.
- print 하지 말고 return 할 것
- 입력 리스트나 딕셔너리를 수정하지 말 것
- 기본값으로 []나 {} 같은 가변 객체를 쓰지 말 것
- 잘못된 입력에는 ValueError를 발생시킬 것
- 정상·경계·잘못된 입력을 포함한 test_ 함수를 함께 작성할 것
```

그리고 받은 테스트만 믿지 말고, **내가 생각한 경계값 테스트를 하나 이상 추가**해 실행합니다. AI가 쓴 테스트는 AI 코드의 실수를 그대로 반영할 수 있기 때문입니다.

---
## 7. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 함수 구성 | 매개변수·기본값·키워드 인자·여러 값 반환·docstring·타입 힌트 |
| print vs return | 계산 함수는 return, 출력은 밖에서 |
| 범위 | 지역 변수는 밖에서 안 보임, 전역 수정은 피하기 |
| 순수 함수 | 같은 입력 → 같은 출력, 입력 불변 → 테스트 쉬움 |
| 잘못된 입력 | `raise ValueError("이유")` |
| 테스트 | `test_` 함수 + `assert`, 정상·경계·잘못된 입력 |
| 모듈 | `%%writefile`, `import`, `importlib.reload`, `if __name__ == "__main__":` |
| AI 검증 | 가변 기본값 `[]` → `None` |

### 자기 점검

- [ ] `print`만 하는 함수가 테스트하기 어려운 이유를 설명할 수 있다.
- [ ] 정상·경계·잘못된 입력을 포함한 테스트 함수를 작성할 수 있다.
- [ ] `%%writefile`로 모듈을 만들고 `import`할 수 있다.
- [ ] `def f(x, items=[])`의 문제를 설명하고 고칠 수 있다.

### 다음 주 예고

**Week 06 · 파일 · CSV · 예외 처리**: 실제 CSV 성적 파일을 읽고, 잘못된 행을 `try/except`로 걸러 내는 견고한 프로그램을 만듭니다.